# Artifact Registry Consolidation

## tl;dr

Rebuilds `output/artifact_index.json` from the actual contents of `Evaluation/output/` so the registry
becomes a *complete* SHA-256 manifest of every produced artifact (data CSVs/JSONs, figures, `.npz` caches
and `.pth` checkpoints), not just the dashboards that the per-phase notebooks registered through
`save_figure`.

Phase notebooks 01–09 write most data tables with raw `pandas.to_csv` / `json.dump` calls, so those files
were never registered at write time. This notebook fixes that gap without re-running the research loop: it
scans the output tree deterministically, derives phase and kind from the folder layout, hashes every file,
and rewrites `artifact_index.json`. Re-running it is idempotent and keeps the index in sync with the tree.

## Rules

- Phase is taken from the top-level folder name under `output/` (e.g. `01_mark_1` → `mark_1`,
  `09_consolidated` → `consolidated`, `10_visualization_hub` → `visualization_hub`).
- Kind is taken from the sub-folder / suffix: `figures/` → `figures`, `caches/` → `caches`,
  `*.pth` → `checkpoints`, everything else → `data`.
- `artifact_index.json` itself is never registered.
- Existing entries are replaced by their on-disk state (same phase + name); stale entries are dropped.
- `version` is bumped to 2 to signal the complete-manifest format.


In [1]:
import hashlib, json, sys
from datetime import datetime, timezone
from pathlib import Path

EVAL_ROOT = Path(r"D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation")
OUTPUT_ROOT = EVAL_ROOT / "output"
INDEX_PATH = OUTPUT_ROOT / "artifact_index.json"

PHASE_FOLDER = {
    "00_pipeline_overview": "00_setup",
    "01_mark_1": "mark_1", "02_mark_2": "mark_2", "03_mark_3": "mark_3",
    "04_mark_4": "mark_4", "05_mark_4b": "mark_4b", "06_mark_4c": "mark_4c",
    "07_mark_4d": "mark_4d", "08_mark_4e": "mark_4e", "09_consolidated": "consolidated",
    "10_visualization_hub": "visualization_hub",
}


def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def classify(relative: Path) -> str:
    parts = relative.parts
    if "figures" in parts:
        return "figures"
    if "caches" in parts:
        return "caches"
    if relative.suffix.lower() == ".pth":
        return "checkpoints"
    return "data"


def build_index() -> list[dict]:
    entries = []
    for folder in sorted(OUTPUT_ROOT.iterdir()):
        if not folder.is_dir():
            continue
        phase = PHASE_FOLDER.get(folder.name, folder.name)
        for file_path in sorted(folder.rglob("*")):
            if not file_path.is_file():
                continue
            relative = file_path.relative_to(folder)
            entries.append({
                "phase": phase,
                "name": str(relative),
                "kind": classify(relative),
                "sha256": sha256_file(file_path),
                "timestamp": datetime.fromtimestamp(file_path.stat().st_mtime, tz=timezone.utc).isoformat(),
            })
    entries.sort(key=lambda a: (a["phase"], a["name"]))
    return entries


In [2]:
# Write the summary tables first so they get captured by the final registry scan.
from collections import Counter
import pandas as pd

summary = Counter((a["phase"], a["kind"]) for a in build_index())
rows = [{"phase": p, "kind": k, "count": c} for (p, k), c in sorted(summary.items())]
df = pd.DataFrame(rows)
out_dir = OUTPUT_ROOT / "11_artifact_registry" / "data"
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / "registry_summary.csv", index=False)

total_by_phase = df.groupby("phase")["count"].sum().sort_index()
print(df.to_string(index=False))
print("\nPer phase:")
print(total_by_phase.to_string())


               phase        kind  count
11_artifact_registry        data      2
        consolidated        data      3
        consolidated     figures      2
              mark_1      caches     13
              mark_1        data      8
              mark_1     figures      5
              mark_2        data      7
              mark_2     figures      5
              mark_3 checkpoints      3
              mark_3        data      7
              mark_3     figures      5
              mark_4        data      8
              mark_4     figures      4
             mark_4b      caches     13
             mark_4b        data      7
             mark_4b     figures      6
             mark_4c checkpoints      2
             mark_4c        data      5
             mark_4c     figures      5
             mark_4d      caches     26
             mark_4d        data      6
             mark_4d     figures      4
             mark_4e        data      6
             mark_4e     figures      4


In [3]:
# Final registry write (after all outputs of this notebook exist), then verify exact sync.
entries = build_index()
index = {"version": 2, "artifact_count": len(entries), "artifacts": entries}
INDEX_PATH.write_text(json.dumps(index, indent=2), encoding="utf-8")

counts = {}
for e in entries:
    counts.setdefault(e["kind"], 0)
    counts[e["kind"]] += 1
print(f"Registered {len(entries)} artifacts -> {INDEX_PATH}")
print("Kinds:", ", ".join(f"{k}={v}" for k, v in sorted(counts.items())))

registered = json.loads(INDEX_PATH.read_text(encoding="utf-8"))["artifacts"]
on_disk = []
for folder in OUTPUT_ROOT.iterdir():
    if not folder.is_dir():
        continue
    phase = PHASE_FOLDER.get(folder.name, folder.name)
    for file_path in folder.rglob("*"):
        if file_path.is_file():
            on_disk.append((phase, str(file_path.relative_to(folder))))
disk_set = set(on_disk)
reg_set = {(a["phase"], a["name"]) for a in registered}

missing = disk_set - reg_set
stale = reg_set - disk_set
print("On-disk files:", len(disk_set))
print("Registered:   ", len(reg_set))
print("Missing from registry:", len(missing))
print("Stale in registry:   ", len(stale))
if missing:
    print("MISSING:", sorted(missing)[:10])
if stale:
    print("STALE:  ", sorted(stale)[:10])

verification = {
    "on_disk_files": len(disk_set),
    "registered_artifacts": len(reg_set),
    "missing_from_registry": sorted(missing)[:20],
    "stale_in_registry": sorted(stale)[:20],
    "complete": not missing and not stale,
}
out_dir = OUTPUT_ROOT / "11_artifact_registry" / "data"
(out_dir / "registry_verification.json").write_text(json.dumps(verification, indent=2), encoding="utf-8")
assert not missing and not stale, "Registry out of sync with output tree"
print("Registry is a complete, exact manifest of the output tree.")


Registered 211 artifacts -> D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\artifact_index.json
Kinds: caches=52, checkpoints=5, data=65, figures=89
On-disk files: 211
Registered:    211
Missing from registry: 0
Stale in registry:    0
Registry is a complete, exact manifest of the output tree.
